# localtuya Config Generator

Reads `tuya_export/tuya_export_*.json` (produced by `export.ipynb`) and generates
a localtuya configuration for all **WiFi devices** (non-sub devices with an IP and a local_key).

Output files are written to `tuya_export/localtuya_*.yaml` and `tuya_export/localtuya_*.json`.

Tested against [rospogrigio/localtuya](https://github.com/rospogrigio/localtuya) v5+.
The YAML format matches what localtuya stores internally in HA config entries.


In [ ]:
%pip install pyyaml --quiet

In [ ]:
import json
import yaml
import datetime
from pathlib import Path
from pprint import pprint

# ── Load the latest export ────────────────────────────────────────────────────
EXPORT_DIR = Path("tuya_export")
export_files = sorted(EXPORT_DIR.glob("tuya_export_*.json"))
if not export_files:
    raise FileNotFoundError(f"No export files in {EXPORT_DIR}. Run export.ipynb first.")

export_path = export_files[-1]   # newest
print(f"Loading: {export_path}")
with open(export_path, encoding="utf-8") as fh:
    data = json.load(fh)

devices_detail    = data["devices"]["detail"]
devices_factory   = data["devices"]["factory"]
devices_spec      = data["devices"]["spec"]        # iot-03, HAS dp_id
devices_functions = data["devices"]["functions"]   # v1, NO dp_id
devices_status    = data["devices"]["status"]
devices_model     = data["devices"].get("model", {})   # v2 thing model
products_funcs    = data.get("products_funcs", {})     # product_id → functions with dp_id

print(f"Devices in export : {len(devices_detail)}")
print(f"With spec         : {len(devices_spec)}")
print(f"With model        : {len(devices_model)}")
print(f"Product functions : {len(products_funcs)} unique product_ids")

# Quick dp_id coverage check
devices_with_dpids = sum(
    1 for did, d in devices_detail.items()
    if not d.get("sub") and (
        any(dp.get("dp_id") for dp in ((devices_spec.get(did) or {}).get("functions") or []))
        or any(f.get("dp_id") for f in (products_funcs.get(d.get("product_id","")) or {}).get("functions") or [])
    )
)
wifi_devices = sum(1 for d in devices_detail.values() if not d.get("sub") and d.get("ip"))
print(f"\nWiFi devices      : {wifi_devices}")
print(f"With dp_ids       : {devices_with_dpids} (from spec or product_funcs)")

## DP classification table

Maps every known Tuya DP code to:
- **platform** — localtuya entity type
- **group** — entities with the same group key are collapsed into one HA entity
  (e.g. all light-related DPs → one `light` entity).  
  For simple entities (switch, sensor, binary_sensor) the group is unique per DP.
- **role** — the field name this DP fills inside its entity (e.g. `brightness`, `color_temp`)
- **default_name** — human-readable fallback name
- optional extra fields passed through to the entity config


In [ ]:
# Each entry: code → dict with keys:
#   platform, group, role, default_name, and any extra localtuya fields
#
# 'role' is the localtuya config key this DP fills inside its entity.
# For the primary/id DP, role="id".
# Groups with the same name share one entity (used for lights, climate, cover).

DP_TABLE = {
    # ── Switches ─────────────────────────────────────────────────────────────
    # Each switch_N is its own entity (unique group per code)
    "switch":          {"platform": "switch", "group": "sw_main",  "role": "id", "default_name": "Switch"},
    "switch_1":        {"platform": "switch", "group": "sw_1",     "role": "id", "default_name": "Switch 1"},
    "switch_2":        {"platform": "switch", "group": "sw_2",     "role": "id", "default_name": "Switch 2"},
    "switch_3":        {"platform": "switch", "group": "sw_3",     "role": "id", "default_name": "Switch 3"},
    "switch_4":        {"platform": "switch", "group": "sw_4",     "role": "id", "default_name": "Switch 4"},
    "switch_5":        {"platform": "switch", "group": "sw_5",     "role": "id", "default_name": "Switch 5"},
    "switch_6":        {"platform": "switch", "group": "sw_6",     "role": "id", "default_name": "Switch 6"},
    "switch_7":        {"platform": "switch", "group": "sw_7",     "role": "id", "default_name": "Switch 7"},
    "switch_8":        {"platform": "switch", "group": "sw_8",     "role": "id", "default_name": "Switch 8"},
    "switch_usb1":     {"platform": "switch", "group": "sw_usb1",  "role": "id", "default_name": "USB 1"},
    "switch_usb2":     {"platform": "switch", "group": "sw_usb2",  "role": "id", "default_name": "USB 2"},
    "switch_usb3":     {"platform": "switch", "group": "sw_usb3",  "role": "id", "default_name": "USB 3"},
    "switch_usb4":     {"platform": "switch", "group": "sw_usb4",  "role": "id", "default_name": "USB 4"},
    "switch_backlight":{"platform": "switch", "group": "sw_backlight", "role": "id", "default_name": "Backlight"},
    "child_lock":      {"platform": "switch", "group": "sw_childlock", "role": "id", "default_name": "Child Lock"},
    "relay_status":    {"platform": "switch", "group": "sw_relay",  "role": "id", "default_name": "Relay"},

    # Power-monitoring DPs attach to switch entities as auxiliary fields.
    # They are also exposed as standalone sensor entities.
    # The 'switch_aux' group is virtual — it's not emitted as its own entity;
    # instead the role values are injected into the switch_1 (or switch) entity.
    "cur_current":        {"platform": "sensor", "group": "sen_cur",    "role": "id",
                           "default_name": "Current",
                           "unit_of_measurement": "mA", "device_class": "current",
                           "state_class": "measurement",
                           "_switch_role": "current"},
    "cur_power":          {"platform": "sensor", "group": "sen_pow",    "role": "id",
                           "default_name": "Power",
                           "unit_of_measurement": "W",  "device_class": "power",
                           "state_class": "measurement",
                           "_switch_role": "current_consumption", "_scale_divisor": 10},
    "cur_voltage":        {"platform": "sensor", "group": "sen_vol",    "role": "id",
                           "default_name": "Voltage",
                           "unit_of_measurement": "V",  "device_class": "voltage",
                           "state_class": "measurement",
                           "_switch_role": "voltage", "_scale_divisor": 10},
    "add_ele":            {"platform": "sensor", "group": "sen_ele",    "role": "id",
                           "default_name": "Energy",
                           "unit_of_measurement": "kWh", "device_class": "energy",
                           "state_class": "total_increasing", "_scale_divisor": 1000},
    "add_ele2":           {"platform": "sensor", "group": "sen_ele2",   "role": "id",
                           "default_name": "Energy 2",
                           "unit_of_measurement": "kWh", "device_class": "energy",
                           "state_class": "total_increasing", "_scale_divisor": 1000},

    # ── Lights ────────────────────────────────────────────────────────────────
    # All same-group DPs → one light entity
    "switch_led":         {"platform": "light", "group": "light",   "role": "id",          "default_name": "Light"},
    "switch_led_1":       {"platform": "light", "group": "light_1", "role": "id",          "default_name": "Light 1"},
    "switch_led_2":       {"platform": "light", "group": "light_2", "role": "id",          "default_name": "Light 2"},
    "work_mode":          {"platform": "light", "group": "light",   "role": "color_mode"},
    "bright_value":       {"platform": "light", "group": "light",   "role": "brightness",  "brightness_lower": 25,  "brightness_upper": 255},
    "bright_value_v2":    {"platform": "light", "group": "light",   "role": "brightness",  "brightness_lower": 10,  "brightness_upper": 1000},
    "colour_data":        {"platform": "light", "group": "light",   "role": "color"},
    "colour_data_v2":     {"platform": "light", "group": "light",   "role": "color"},
    "temp_value":         {"platform": "light", "group": "light",   "role": "color_temp",  "color_temp_min_kelvin": 2700, "color_temp_max_kelvin": 6500},
    "temp_value_v2":      {"platform": "light", "group": "light",   "role": "color_temp",  "color_temp_min_kelvin": 2700, "color_temp_max_kelvin": 6500},

    # ── Environmental sensors ─────────────────────────────────────────────────
    "va_temperature":     {"platform": "sensor", "group": "sen_temp", "role": "id",
                           "default_name": "Temperature",
                           "unit_of_measurement": "°C", "device_class": "temperature",
                           "state_class": "measurement", "_scale_divisor": 10},
    "temp_current":       {"platform": "sensor", "group": "sen_temp", "role": "id",
                           "default_name": "Temperature",
                           "unit_of_measurement": "°C", "device_class": "temperature",
                           "state_class": "measurement"},
    "va_humidity":        {"platform": "sensor", "group": "sen_hum",  "role": "id",
                           "default_name": "Humidity",
                           "unit_of_measurement": "%",  "device_class": "humidity",
                           "state_class": "measurement"},
    "humidity_value":     {"platform": "sensor", "group": "sen_hum",  "role": "id",
                           "default_name": "Humidity",
                           "unit_of_measurement": "%",  "device_class": "humidity",
                           "state_class": "measurement"},
    "illuminance_value":  {"platform": "sensor", "group": "sen_lux",  "role": "id",
                           "default_name": "Illuminance",
                           "unit_of_measurement": "lx", "device_class": "illuminance",
                           "state_class": "measurement"},
    "battery_percentage": {"platform": "sensor", "group": "sen_bat",  "role": "id",
                           "default_name": "Battery",
                           "unit_of_measurement": "%",  "device_class": "battery",
                           "state_class": "measurement"},
    "pm25_value":         {"platform": "sensor", "group": "sen_pm25", "role": "id",
                           "default_name": "PM2.5",
                           "unit_of_measurement": "µg/m³", "device_class": "pm25",
                           "state_class": "measurement"},
    "co2_value":          {"platform": "sensor", "group": "sen_co2",  "role": "id",
                           "default_name": "CO2",
                           "unit_of_measurement": "ppm", "device_class": "carbon_dioxide",
                           "state_class": "measurement"},
    "voc_value":          {"platform": "sensor", "group": "sen_voc",  "role": "id",
                           "default_name": "VOC",
                           "unit_of_measurement": "µg/m³",
                           "state_class": "measurement"},

    # ── Binary sensors ────────────────────────────────────────────────────────
    "pir":                     {"platform": "binary_sensor", "group": "bs_pir",   "role": "id",
                                "default_name": "Motion",    "device_class": "motion",
                                "state_on": "pir", "state_off": "none"},
    "presence_state":          {"platform": "binary_sensor", "group": "bs_pres",  "role": "id",
                                "default_name": "Presence",  "device_class": "presence",
                                "state_on": "presence", "state_off": "none"},
    "smoke_sensor_status":     {"platform": "binary_sensor", "group": "bs_smoke", "role": "id",
                                "default_name": "Smoke",     "device_class": "smoke",
                                "state_on": "alarm", "state_off": "normal"},
    "gas_sensor_status":       {"platform": "binary_sensor", "group": "bs_gas",   "role": "id",
                                "default_name": "Gas",       "device_class": "gas",
                                "state_on": "alarm", "state_off": "normal"},
    "water_sensor_state":      {"platform": "binary_sensor", "group": "bs_water", "role": "id",
                                "default_name": "Water Leak", "device_class": "moisture",
                                "state_on": "alarm", "state_off": "normal"},
    "door_contact_state":      {"platform": "binary_sensor", "group": "bs_door",  "role": "id",
                                "default_name": "Door",      "device_class": "door",
                                "state_on": "open", "state_off": "close"},
    "mcs_door_contact_state":  {"platform": "binary_sensor", "group": "bs_door",  "role": "id",
                                "default_name": "Door",      "device_class": "door",
                                "state_on": "open", "state_off": "close"},
    "fault":                   {"platform": "binary_sensor", "group": "bs_fault", "role": "id",
                                "default_name": "Fault",     "device_class": "problem",
                                "state_on": "1", "state_off": "0"},

    # ── Climate ───────────────────────────────────────────────────────────────
    # All climate DPs share one entity group
    "temp_set":    {"platform": "climate", "group": "climate", "role": "target_temperature_dp", "default_name": "Thermostat"},
    "temp_set_f":  {"platform": "climate", "group": "climate", "role": "target_temperature_dp", "default_name": "Thermostat"},
    "mode":        {"platform": "climate", "group": "climate", "role": "hvac_mode_dp"},
    "work_state":  {"platform": "climate", "group": "climate", "role": "hvac_action_dp"},
    "eco":         {"platform": "climate", "group": "climate", "role": "eco_dp"},
    "fan_speed_enum": {"platform": "climate", "group": "climate", "role": "hvac_fan_mode_dp"},

    # ── Cover / blinds ────────────────────────────────────────────────────────
    "control":         {"platform": "cover", "group": "cover", "role": "id", "default_name": "Cover",
                        "commands_set": "open|stop|close", "positioning_mode": "none"},
    "percent_control": {"platform": "cover", "group": "cover", "role": "set_position_dp"},
    "percent_state":   {"platform": "cover", "group": "cover", "role": "current_position_dp"},
    "position":        {"platform": "cover", "group": "cover", "role": "set_position_dp"},

    # ── Fan ───────────────────────────────────────────────────────────────────
    "switch_fan":     {"platform": "fan", "group": "fan", "role": "id", "default_name": "Fan"},
    "fan_speed":      {"platform": "fan", "group": "fan", "role": "fan_speed_control"},
    "fan_direction":  {"platform": "fan", "group": "fan", "role": "fan_direction"},
}

# Keys that are internal meta-hints, not real localtuya config keys
_META_KEYS = {"platform", "group", "role", "default_name", "_switch_role", "_scale_divisor"}

# Platforms where all DPs in the same group collapse into ONE entity
COMPOUND_PLATFORMS = {"light", "climate", "cover", "fan"}

print(f"DP table has {len(DP_TABLE)} known codes.")

## Helper functions

In [ ]:
def parse_values(v):
    """Tuya 'values' field is a JSON-encoded string inside the outer JSON."""
    if not v:
        return {}
    if isinstance(v, dict):
        return v
    try:
        return json.loads(v)
    except Exception:
        return {}


def humanize(code):
    """'switch_1' → 'Switch 1'"""
    return code.replace("_", " ").title()


def _apply_dp_list(dp_map, dp_list, source, prefer_name=False):
    """
    Merge a list of DP dicts into dp_map.
    Each dp dict must have 'code'. Optional: dp_id, type, name/custom_name, values.
    If a code already exists in dp_map, only backfill missing dp_id / enrich name.
    """
    for dp in (dp_list or []):
        code = dp.get("code", "")
        if not code:
            continue
        dp_id  = dp.get("dp_id") or dp.get("id")
        name   = (dp.get("custom_name") or "").strip() or dp.get("name") or dp.get("desc") or ""
        values = parse_values(dp.get("values") or dp.get("value_desc") or "")
        dtype  = dp.get("type", "")

        if code not in dp_map:
            dp_map[code] = {
                "dp_id":  dp_id,
                "type":   dtype,
                "name":   name or humanize(code),
                "values": values,
                "source": source,
            }
        else:
            entry = dp_map[code]
            # Backfill dp_id if still missing
            if entry["dp_id"] is None and dp_id:
                entry["dp_id"] = dp_id
                entry["source"] = source  # note where dp_id came from
            # Enrich name if we have a better one
            if prefer_name and name and entry["name"] == humanize(code):
                entry["name"] = name
            # Enrich values if empty
            if not entry["values"] and values:
                entry["values"] = values


def get_dp_map(device_id):
    """
    Build {code: {dp_id, type, name, values, source}} for a device.

    dp_id lookup priority (highest → lowest):
      0. LOCAL_DP_IDS[device_id]              — tinytuya LAN poll, ground truth (overrides all)
      1. devices_spec[device_id]              — iot-03 per-device spec, has dp_id + custom_name
      2. products_funcs[product_id]           — product-level functions, reliable dp_id source
      3. devices_functions[device_id]         — v1 functions, NO dp_id but has user-visible names
      4. devices_model[device_id]             — v2 thing model, may have dp_id
      5. devices_status[device_id]            — live status, NO dp_id (code presence only)
    """
    dp_map = {}
    product_id = devices_detail.get(device_id, {}).get("product_id", "")

    # 1. iot-03 device specification (best source: has dp_id AND custom_name)
    spec = devices_spec.get(device_id, {})
    for section in ("functions", "status"):
        _apply_dp_list(dp_map, spec.get(section), source="spec")

    # 2. Product-level functions (reliable dp_id when spec is empty or dp_id is null)
    if product_id and product_id in products_funcs:
        prod_funcs = (products_funcs[product_id] or {}).get("functions") or []
        _apply_dp_list(dp_map, prod_funcs, source="product_funcs")

    # 3. v1 functions endpoint — NO dp_id, but has the best user-visible DP names
    funcs = devices_functions.get(device_id, {})
    _apply_dp_list(dp_map, funcs.get("functions"), source="functions", prefer_name=True)

    # 4. v2 thing model — parse model JSON for dp_id if available
    model_raw = devices_model.get(device_id)
    if model_raw:
        # model field is typically a JSON string; it may contain 'properties' list
        if isinstance(model_raw, str):
            try:
                model_raw = json.loads(model_raw)
            except Exception:
                model_raw = {}
        # Try to find a properties/services list
        for path in (["properties"], ["services", 0, "properties"], ["schema"]):
            node = model_raw
            try:
                for key in path:
                    if isinstance(key, int):
                        node = node[key]
                    else:
                        node = node[key]
                if isinstance(node, list):
                    _apply_dp_list(dp_map, node, source="model")
                    break
            except (KeyError, IndexError, TypeError):
                continue

    # 5. Live status — just ensures codes are present even without any spec
    for dp in (devices_status.get(device_id) or []):
        code = dp.get("code", "")
        if code and code not in dp_map:
            dp_map[code] = {
                "dp_id":  None,
                "type":   type(dp.get("value", "")).__name__,
                "name":   humanize(code),
                "values": {},
                "source": "status_only",
            }

    # 0. Local LAN polling via tinytuya — apply as highest-priority override
    #    LOCAL_DP_IDS is populated by the tinytuya polling cell below.
    #    Using globals().get() so helpers cell can be run before the polling cell.
    local_ids = globals().get("LOCAL_DP_IDS", {}).get(device_id, {})
    for code, dp_id in local_ids.items():
        if code in dp_map:
            dp_map[code]["dp_id"] = dp_id
            dp_map[code]["source"] = "local_lan"
        else:
            # Code only found locally — add a minimal entry
            dp_map[code] = {
                "dp_id":  dp_id,
                "type":   "unknown",
                "name":   humanize(code),
                "values": {},
                "source": "local_lan",
            }

    return dp_map


def get_local_key(device_id):
    """Extract local_key from factory info, fall back to device detail."""
    fi = devices_factory.get(device_id)
    if fi:
        item = fi[0] if isinstance(fi, list) and fi else fi
        if isinstance(item, dict):
            key = item.get("local_key", "")
            if key:
                return key
    return devices_detail.get(device_id, {}).get("local_key", "")


def scaling_for_dp(code, dp_info, dp_table_entry):
    """
    Compute localtuya 'scaling' multiplier for a sensor entity.
    Tuya 'scale' field means: actual_value = raw / 10^scale
    localtuya 'scaling' is a plain multiplier: e.g. 0.1 means divide by 10.
    """
    vals = dp_info.get("values", {})
    tuya_scale = vals.get("scale", 0)
    fixed_div  = dp_table_entry.get("_scale_divisor", 1)
    total_divisor = (10 ** tuya_scale) * fixed_div
    return round(1.0 / total_divisor, 6) if total_divisor != 1 else None


print("Helpers defined.")

# ── Quick diagnostic: show dp_id coverage for all WiFi devices ────────────────
print("\nDP-id coverage:")
print(f"  {'Device':<30} {'product_id':<16} {'codes':>6} {'with_dp_id':>10} {'sources'}")
print("  " + "-" * 80)
for did, d in sorted(devices_detail.items(), key=lambda x: x[1].get("name","").lower()):
    if d.get("sub") or not d.get("ip"):
        continue
    dm = get_dp_map(did)
    n_total = len(dm)
    n_with  = sum(1 for v in dm.values() if v["dp_id"] is not None)
    sources = sorted({v["source"] for v in dm.values()})
    flag    = "" if n_with == n_total else f"  ← {n_total - n_with} missing"
    print(f"  {d.get('name','?'):<30} {d.get('product_id',''):<16} {n_total:>6} {n_with:>10}  {sources}{flag}")

In [ ]:
# LOCAL_DP_IDS is populated by the tinytuya polling cell below.
# get_dp_map() checks it first as the highest-priority source.
LOCAL_DP_IDS = {}   # device_id -> {code: dp_id_int}

## Local IP discovery

The Tuya cloud export stores the **external / NAT IP** — useless for LAN control.
This cell finds the real local IPs using two complementary methods:

1. **tinytuya UDP broadcast** — sends a broadcast on port 6666/6667 and waits for
   devices to reply. Each reply contains the `gwId` (= device_id) and the device's
   local IP, so the mapping is automatic. Fast (~3 s).

2. **TCP port-6668 scan** (fallback) — scans the entire subnet for Tuya's local-control
   port. For any hit not yet resolved, we probe it with the known `device_id`/`local_key`
   from the export; a valid `dps` response confirms the identity.

Result is stored in `LOCAL_IP_MAP = {device_id: "10.0.1.x"}`.
The polling cell below reads this dict and uses the local IP instead of the cloud IP.


In [ ]:
import tinytuya
import socket
import ipaddress
import concurrent.futures
import subprocess
import platform
import time

# ── Configure your LAN here ───────────────────────────────────────────────────
LAN_SUBNET        = "10.0.1.0/24"   # ← your LAN subnet
UDP_SCAN_RETRIES  = 20              # broadcast rounds — some devices broadcast every 20-30 s
PING_TIMEOUT      = 1.0             # seconds, per host
PING_WORKERS      = 200             # parallel ping threads
TCP_TIMEOUT       = 1.0             # TCP probe timeout per host (longer = more reliable)
TCP_WORKERS       = 128             # parallel threads for port sweep
PROBE_VERSIONS    = ["3.3", "3.1", "3.4"]  # try all protocol versions when matching
PROBE_TIMEOUT     = 3               # tinytuya handshake timeout

# ─────────────────────────────────────────────────────────────────────────────
LOCAL_IP_MAP = {}   # device_id → local LAN IP

# ── Step 1: tinytuya UDP broadcast scan ───────────────────────────────────────
# Devices announce themselves periodically; more retries = longer listen window.
print(f"Step 1: UDP broadcast scan (maxretry={UDP_SCAN_RETRIES}, ~{UDP_SCAN_RETRIES*0.3:.0f} s)…")
try:
    found_udp = tinytuya.deviceScan(verbose=False, maxretry=UDP_SCAN_RETRIES, color=False)
    print(f"  Found {len(found_udp)} device(s) via UDP:")
    for gw_id, info in found_udp.items():
        local_ip = info.get("ip", "")
        ver      = info.get("version", "?")
        name     = devices_detail.get(gw_id, {}).get("name", gw_id)
        if gw_id in devices_detail:
            LOCAL_IP_MAP[gw_id] = local_ip
            print(f"    ✓ {name!r:<35} {local_ip}  v{ver}")
        else:
            print(f"    ? unknown gwId {gw_id}  {local_ip}  v{ver}  (not in export)")
except Exception as e:
    print(f"  UDP scan error: {e}")

# ── Step 2: Ping sweep to find all live hosts ─────────────────────────────────
# Much faster than blind TCP scan — eliminates dead IPs before port probing.
missing_devices = {
    did: d for did, d in devices_detail.items()
    if not d.get("sub") and did not in LOCAL_IP_MAP and get_local_key(did)
}

_system = platform.system()
def _ping(ip_str):
    try:
        if _system == "Darwin":
            cmd = ["ping", "-c", "1", "-t", str(int(PING_TIMEOUT)), ip_str]
        elif _system == "Windows":
            cmd = ["ping", "-n", "1", "-w", str(int(PING_TIMEOUT * 1000)), ip_str]
        else:  # Linux
            cmd = ["ping", "-c", "1", "-W", str(int(PING_TIMEOUT)), ip_str]
        r = subprocess.run(cmd, capture_output=True, timeout=PING_TIMEOUT + 1)
        return ip_str if r.returncode == 0 else None
    except Exception:
        return None

net = ipaddress.ip_network(LAN_SUBNET, strict=False)
all_hosts = [str(h) for h in net.hosts()]

print(f"\nStep 2: Ping sweep of {LAN_SUBNET} ({len(all_hosts)} hosts)…")
with concurrent.futures.ThreadPoolExecutor(max_workers=PING_WORKERS) as ex:
    live_hosts = [ip for ip in ex.map(_ping, all_hosts) if ip]
print(f"  Live hosts: {len(live_hosts)}  →  {live_hosts}")

# ── Step 3: TCP port-6668 scan on live hosts only ─────────────────────────────
claimed = set(LOCAL_IP_MAP.values())

def _probe_port(ip_str):
    try:
        with socket.create_connection((ip_str, 6668), timeout=TCP_TIMEOUT):
            return ip_str
    except Exception:
        return None

print(f"\nStep 3: TCP scan port 6668 on {len(live_hosts)} live host(s)…")
with concurrent.futures.ThreadPoolExecutor(max_workers=TCP_WORKERS) as ex:
    open_ips = [ip for ip in ex.map(_probe_port, live_hosts) if ip]

candidate_ips = [ip for ip in open_ips if ip not in claimed]
print(f"  Port 6668 open: {open_ips}")
print(f"  Unclaimed candidates: {candidate_ips}")

# ── Step 4: Match candidates to device_ids via tinytuya handshake ─────────────
if candidate_ips and missing_devices:
    print(f"\nStep 4: Probing {len(missing_devices)} device(s) × "
          f"{len(candidate_ips)} IP(s) × {len(PROBE_VERSIONS)} version(s)…")
    for did, detail in missing_devices.items():
        local_key = get_local_key(did)
        name      = detail.get("name", did)
        matched   = False
        for ip in candidate_ips:
            if ip in claimed:
                continue
            for ver in PROBE_VERSIONS:
                try:
                    d = tinytuya.Device(did, ip, local_key)
                    d.set_version(float(ver))
                    d.set_socketTimeout(PROBE_TIMEOUT)
                    status = d.status()
                    if status and "dps" in status:
                        LOCAL_IP_MAP[did] = ip
                        claimed.add(ip)
                        print(f"    ✓ {name!r:<35} → {ip}  (v{ver})")
                        matched = True
                        break
                except Exception:
                    pass
            if matched:
                break
        if not matched:
            print(f"    ✗ {name!r:<35}  not found on LAN")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nLOCAL_IP_MAP: {len(LOCAL_IP_MAP)}/{len(missing_devices) + len(LOCAL_IP_MAP)} device(s) resolved.")
for did, ip in sorted(LOCAL_IP_MAP.items(), key=lambda x: devices_detail.get(x[0], {}).get("name", "")):
    name = devices_detail.get(did, {}).get("name", did)
    print(f"  {name!r:<35} {ip}")

## Local LAN polling with tinytuya

Polls every WiFi device directly over the local network to obtain the **numeric dp_ids**
that the Tuya cloud API refuses to expose for Smart Home devices.

**How it works:**
1. `tinytuya.Device.status()` returns `{"dps": {"1": true, "2": 100, ...}}` — raw dp_id → value.
2. The cloud export already has `devices_status[device_id]` → `[{"code": "switch_1", "value": true}, ...]`.
3. We cross-reference the two by value-matching:
   - Unique values (e.g. `100 W`) → direct 1-to-1 match.
   - Repeated values (e.g. multiple `true` booleans) → matched by sorted order (dp_id order vs. cloud status order).
4. The result is stored in `LOCAL_DP_IDS[device_id] = {code: dp_id_int}`.

After running this cell, **re-run the helpers cell** (DP-id coverage) and the main
conversion loop to apply the updated mapping.

> **Prerequisites:** device must be on the same LAN as your machine.  
> Devices that are off or unreachable will be skipped (TIMEOUT/ERROR).


In [ ]:
import tinytuya
import time
from collections import defaultdict

# ── Tuning knobs ──────────────────────────────────────────────────────────────
POLL_TIMEOUT = 6      # TCP socket timeout per attempt (seconds)
POLL_RETRIES = 2      # how many times to retry a failed device
POLL_INTER   = 0.3    # pause between devices (seconds)

# ─────────────────────────────────────────────────────────────────────────────

def _poll_device_local(device_id, ip, local_key, version="3.3"):
    for attempt in range(POLL_RETRIES):
        try:
            d = tinytuya.Device(device_id, ip, local_key)
            d.set_version(float(version))
            d.set_socketTimeout(POLL_TIMEOUT)
            status = d.status()
            if status and "dps" in status:
                return status
            err = status.get("Error") if status else "no response"
            if attempt == POLL_RETRIES - 1:
                print(f"[no dps] {err}")
        except Exception as e:
            if attempt == POLL_RETRIES - 1:
                print(f"[ERR] {e}")
        if attempt < POLL_RETRIES - 1:
            time.sleep(1)
    return None


def _map_codes_to_dpids(local_dps, cloud_status_list):
    """Cross-reference local {dp_id_str: value} → cloud [{code, value}]."""
    local = {int(k): v for k, v in local_dps.items()}
    cloud_ordered = [(item["code"], item["value"]) for item in (cloud_status_list or [])]

    code_to_dpid = {}
    used_dpids, used_codes = set(), set()
    val_to_dpids = defaultdict(list)
    val_to_codes = defaultdict(list)

    for dp_id, val in sorted(local.items()):
        val_to_dpids[repr(val)].append(dp_id)
    for code, val in cloud_ordered:
        val_to_codes[repr(val)].append(code)

    # Pass 1: unique value pairs → direct 1-to-1 match
    for val_key, dp_ids in val_to_dpids.items():
        codes = val_to_codes.get(val_key, [])
        if len(dp_ids) == 1 and len(codes) == 1:
            dp_id, code = dp_ids[0], codes[0]
            if dp_id not in used_dpids and code not in used_codes:
                code_to_dpid[code] = dp_id
                used_dpids.add(dp_id); used_codes.add(code)

    # Pass 2: same-value groups — sort dp_ids numerically, zip with cloud order
    for val_key, dp_ids in val_to_dpids.items():
        free_dpids = [d for d in dp_ids if d not in used_dpids]
        free_codes = [c for c in val_to_codes.get(val_key, []) if c not in used_codes]
        for dp_id, code in zip(sorted(free_dpids), free_codes):
            code_to_dpid[code] = dp_id
            used_dpids.add(dp_id); used_codes.add(code)

    return code_to_dpid


# ── Main polling loop ─────────────────────────────────────────────────────────
_local_ip_map = globals().get("LOCAL_IP_MAP", {})

if not _local_ip_map:
    print("LOCAL_IP_MAP is empty — run the IP discovery cell first.")
else:
    print(f"Polling {len(_local_ip_map)} LAN-reachable device(s) for dp_ids…\n")

LOCAL_DP_IDS = {}

for device_id, detail in sorted(devices_detail.items(), key=lambda x: x[1].get("name", "")):
    if detail.get("sub"):
        continue
    if device_id not in _local_ip_map:
        continue   # not found on LAN — skip entirely

    local_key = get_local_key(device_id)
    if not local_key:
        continue

    ip      = _local_ip_map[device_id]
    name    = detail.get("name", device_id)
    version = globals().get("PROTOCOL_OVERRIDES", {}).get(device_id,
              globals().get("PROTOCOL_DEFAULT", "3.3"))

    print(f"  {name!r:35s} {ip:16s} v{version}  … ", end="", flush=True)
    status = _poll_device_local(device_id, ip, local_key, version)

    if not status:
        print("TIMEOUT / ERROR")
        time.sleep(POLL_INTER)
        continue

    local_dps    = status.get("dps", {})
    cloud_status = devices_status.get(device_id, [])
    mapping      = _map_codes_to_dpids(local_dps, cloud_status)
    LOCAL_DP_IDS[device_id] = mapping

    n_local   = len(local_dps)
    n_matched = len(mapping)
    unmatched = n_local - n_matched
    flag = f"  ← {unmatched} dp_ids unmatched" if unmatched else ""
    print(f"OK — {n_local} raw DPs, {n_matched} codes mapped{flag}")
    for code, dp_id in sorted(mapping.items(), key=lambda x: x[1]):
        print(f"      dp {dp_id:>3} → {code}")

    time.sleep(POLL_INTER)

print(f"\nDone. LOCAL_DP_IDS populated for {len(LOCAL_DP_IDS)} device(s).")
print("→ Re-run the helpers cell and the main loop to apply the mapping.")

## Entity builders

In [ ]:
def build_entities(device_name, dp_map):
    """
    Given a dp_map ({code: {dp_id, type, name, values}}),
    return a list of localtuya entity dicts.

    Strategy:
    1. Ignore codes not in DP_TABLE.
    2. Group codes by (platform, group) key.
    3. For compound platforms (light, climate, cover, fan): one entity per group,
       each DP fills a named role field.
    4. For simple platforms (switch, sensor, binary_sensor): one entity per code.
    5. Power-monitoring DPs (cur_current, cur_power, cur_voltage) are:
       - Emitted as standalone sensor entities
       - Their dp_id is also injected into the main switch entity (if a switch exists)
    """
    warnings = []

    # ── Step 1: Classify all DPs ──────────────────────────────────────────────
    groups = {}  # (platform, group_key) -> list of {code, dp_id, dp_info, table_entry}

    for code, dp_info in dp_map.items():
        if code not in DP_TABLE:
            continue
        te = DP_TABLE[code]
        key = (te["platform"], te["group"])
        groups.setdefault(key, []).append({
            "code":    code,
            "dp_id":   dp_info["dp_id"],
            "dp_info": dp_info,
            "te":      te,
        })

    # ── Step 2: Collect power-monitoring dp_ids for switch injection ───────────
    pm_switch_fields = {}  # localtuya switch role → dp_id
    for code, dp_info in dp_map.items():
        if code in DP_TABLE:
            te = DP_TABLE[code]
            role = te.get("_switch_role")
            if role and dp_info["dp_id"] is not None:
                pm_switch_fields[role] = dp_info["dp_id"]

    # ── Step 3: Determine if power monitoring goes into switch or stays sensor ─
    # Find the "main" switch (sw_1 or sw_main) for PM injection
    main_switch_group_key = None
    for gkey in ("sw_1", "sw_main", "sw_relay"):
        if ("switch", gkey) in groups:
            main_switch_group_key = gkey
            break

    # ── Step 4: Build entities ────────────────────────────────────────────────
    entities = []

    for (platform, group_key), members in sorted(groups.items()):
        if platform in COMPOUND_PLATFORMS:
            # ── Compound entity: merge all roles into one dict ─────────────────
            # Find the primary (role="id") DP
            primary = next((m for m in members if m["te"]["role"] == "id"), members[0])
            entity = {}

            # Friendly name: device_name + DP name for the primary DP
            pname = primary["dp_info"]["name"] or primary["te"].get("default_name", "")
            entity["friendly_name"] = f"{device_name} {pname}".strip()
            entity["platform"] = platform

            # Primary dp_id → "id"
            pid = primary["dp_id"]
            entity["id"] = pid if pid is not None else "# FILL_DP_ID"
            if pid is None:
                warnings.append(f"[{platform}/{group_key}] dp_id unknown for '{primary['code']}'")

            # Secondary roles
            for m in members:
                role = m["te"]["role"]
                if role == "id":
                    continue
                mid = m["dp_id"]
                entity[role] = mid if mid is not None else "# FILL_DP_ID"
                if mid is None:
                    warnings.append(f"[{platform}/{group_key}] dp_id unknown for '{m['code']}'")

            # Inject extra static fields from DP_TABLE (e.g. brightness_lower, commands_set)
            for m in members:
                for k, v in m["te"].items():
                    if k not in _META_KEYS and k not in entity:
                        entity[k] = v

            # Derive brightness range from Tuya spec values
            if platform == "light":
                for m in members:
                    if m["te"]["role"] == "brightness":
                        vals = m["dp_info"]["values"]
                        if vals.get("min") is not None:
                            entity["brightness_lower"] = vals["min"]
                        if vals.get("max") is not None:
                            entity["brightness_upper"] = vals["max"]

            entities.append(entity)

        elif platform == "switch":
            # ── Switch: one entity per group ──────────────────────────────────
            m = members[0]  # exactly one member per switch group
            dp_id = m["dp_id"]
            name  = m["dp_info"]["name"] or m["te"].get("default_name", "")
            entity = {
                "platform":      "switch",
                "friendly_name": f"{device_name} {name}".strip(),
                "id":            dp_id if dp_id is not None else "# FILL_DP_ID",
            }
            if dp_id is None:
                warnings.append(f"[switch/{group_key}] dp_id unknown for '{m['code']}'")

            # Inject PM fields into the main switch
            if group_key == main_switch_group_key and pm_switch_fields:
                entity.update(pm_switch_fields)

            entities.append(entity)

        elif platform == "sensor":
            # ── Sensor: one entity per group ──────────────────────────────────
            m = members[0]
            dp_id = m["dp_id"]
            te    = m["te"]
            name  = m["dp_info"]["name"] or te.get("default_name", "")
            entity = {
                "platform":      "sensor",
                "friendly_name": f"{device_name} {name}".strip(),
                "id":            dp_id if dp_id is not None else "# FILL_DP_ID",
            }
            if dp_id is None:
                warnings.append(f"[sensor/{group_key}] dp_id unknown for '{m['code']}'")

            # Static fields (unit, device_class, state_class)
            for k in ("unit_of_measurement", "device_class", "state_class"):
                if k in te:
                    entity[k] = te[k]

            # Scaling
            sc = scaling_for_dp(m["code"], m["dp_info"], te)
            if sc is not None:
                entity["scaling"] = sc

            entities.append(entity)

        elif platform == "binary_sensor":
            # ── Binary sensor: one entity per group ───────────────────────────
            m = members[0]
            dp_id = m["dp_id"]
            te    = m["te"]
            name  = m["dp_info"]["name"] or te.get("default_name", "")
            entity = {
                "platform":      "binary_sensor",
                "friendly_name": f"{device_name} {name}".strip(),
                "id":            dp_id if dp_id is not None else "# FILL_DP_ID",
            }
            if dp_id is None:
                warnings.append(f"[binary_sensor/{group_key}] dp_id unknown for '{m['code']}'")
            for k in ("device_class", "state_on", "state_off"):
                if k in te:
                    entity[k] = te[k]
            entities.append(entity)

    return entities, warnings


print("Entity builder defined.")

## Main conversion loop

In [ ]:
# ── Optional overrides ────────────────────────────────────────────────────────
# Set the protocol version for specific devices, or a global default.
# Most modern devices use 3.3. Older devices may use 3.1. Very new ones: 3.4.
PROTOCOL_DEFAULT = "3.3"
PROTOCOL_OVERRIDES = {
    # "<device_id>": "3.4",
}

# Set to skip specific device IDs
SKIP_DEVICE_IDS = set()

# ─────────────────────────────────────────────────────────────────────────────
# Use locally-discovered IPs where available (LOCAL_IP_MAP from the discovery cell).
# Falls back to the cloud export IP — but that's usually the external/NAT address.
_local_ip_map = globals().get("LOCAL_IP_MAP", {})

localtuya_devices = []   # list of device config dicts
all_warnings = {}        # device_id -> [warning strings]
skipped = []

for device_id, detail in sorted(devices_detail.items(), key=lambda x: x[1].get("name", "")):
    if device_id in SKIP_DEVICE_IDS:
        skipped.append((device_id, "in SKIP_DEVICE_IDS"))
        continue

    # localtuya only handles WiFi devices (sub=False, has IP)
    if detail.get("sub", False):
        skipped.append((device_id, f"sub-device (node_id={detail.get('node_id','')}, gw={detail.get('gateway_id','')})"))
        continue

    # Prefer the locally-resolved LAN IP; fall back to cloud export IP
    ip = _local_ip_map.get(device_id) or detail.get("ip", "")
    if not ip:
        skipped.append((device_id, "no IP address (offline or gateway-only)"))
        continue

    local_key = get_local_key(device_id)
    if not local_key:
        skipped.append((device_id, "no local_key (factory-infos not available)"))
        continue

    device_name = detail.get("name", device_id)
    category    = detail.get("category", "?")
    dp_map      = get_dp_map(device_id)

    if not dp_map:
        skipped.append((device_id, f"{device_name!r}: no DP info available (spec/functions empty)"))
        continue

    entities, warnings = build_entities(device_name, dp_map)

    if not entities:
        skipped.append((device_id, f"{device_name!r} cat={category}: no recognised DPs"))
        continue

    if warnings:
        all_warnings[device_id] = warnings

    proto = PROTOCOL_OVERRIDES.get(device_id, PROTOCOL_DEFAULT)

    dev_cfg = {
        "host":             ip,
        "device_id":        device_id,
        "local_key":        local_key,
        "friendly_name":    device_name,
        "protocol_version": proto,
        "model":            detail.get("model", ""),
        "product_key":      detail.get("product_id", ""),
        "entities":         entities,
    }
    localtuya_devices.append(dev_cfg)

print(f"Generated configs for {len(localtuya_devices)} WiFi devices.")
print(f"Skipped {len(skipped)} devices.")
print(f"Devices with warnings: {len(all_warnings)}")
if _local_ip_map:
    print(f"Using LAN IPs for {sum(1 for d in localtuya_devices if d['host'] in _local_ip_map.values())} device(s).")

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 80)
print("LOCALTUYA DEVICE SUMMARY")
print("=" * 80)

for dev in localtuya_devices:
    print(f"\n{'─'*60}")
    print(f"  {dev['friendly_name']!r}")
    print(f"  device_id : {dev['device_id']}")
    print(f"  host      : {dev['host']}")
    print(f"  local_key : {dev['local_key'][:8]}...")
    print(f"  protocol  : {dev['protocol_version']}")
    print(f"  product   : {dev['product_key']}")
    print(f"  entities ({len(dev['entities'])}):")
    for e in dev["entities"]:
        extras = {k: v for k, v in e.items()
                  if k not in ("platform", "friendly_name", "id")}
        print(f"    [{e['platform']:<14}] dp={e['id']:<5} {e['friendly_name']!r}")
        if extras:
            for k, v in extras.items():
                print(f"      {k}: {v}")
    if dev["device_id"] in all_warnings:
        print("  WARNINGS:")
        for w in all_warnings[dev["device_id"]]:
            print(f"    ⚠  {w}")

print(f"\n{'─'*60}")
print(f"\nSKIPPED ({len(skipped)}):")
for did, reason in skipped:
    print(f"  {did}: {reason}")

## Review & patch before saving

Edit `localtuya_devices` here before saving — e.g. fix protocol version,
rename entities, fill in missing dp_ids, adjust brightness ranges.


In [ ]:
# Example patches — uncomment and adapt as needed:

# # Rename an entity
# for dev in localtuya_devices:
#     if dev["device_id"] == "<id>":
#         for e in dev["entities"]:
#             if e["id"] == 1:
#                 e["friendly_name"] = "Living Room Socket Switch 1"

# # Fix protocol version for a specific device
# for dev in localtuya_devices:
#     if dev["device_id"] == "<id>":
#         dev["protocol_version"] = "3.4"

# # Fill a missing dp_id manually
# for dev in localtuya_devices:
#     if dev["device_id"] == "<id>":
#         for e in dev["entities"]:
#             if e.get("id") == "# FILL_DP_ID":
#                 e["id"] = 7

print("Patch cell executed (no changes unless uncommented).")

## Save output

In [ ]:
import datetime
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# ── JSON (exact format localtuya stores in HA config entries) ─────────────────
json_path = EXPORT_DIR / f"localtuya_{ts}.json"
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(localtuya_devices, fh, indent=2, ensure_ascii=False)
print(f"JSON → {json_path}")

# ── YAML (configuration.yaml format for localtuya < 4.0 and documentation) ───
# Produce a clean YAML with all numeric dp_ids (yaml.dump preserves int types)
yaml_path = EXPORT_DIR / f"localtuya_{ts}.yaml"
yaml_doc  = {"localtuya": localtuya_devices}
with open(yaml_path, "w", encoding="utf-8") as fh:
    yaml.dump(yaml_doc, fh, allow_unicode=True, default_flow_style=False,
              sort_keys=False, indent=2)
print(f"YAML → {yaml_path}")

# ── Markdown summary for quick reference ─────────────────────────────────────
md_lines = ["# localtuya Device Config Summary\n",
            f"Generated: {ts}  |  Source: {export_path.name}\n"]
for dev in localtuya_devices:
    md_lines.append(f"\n## {dev['friendly_name']}\n")
    md_lines.append(f"- device_id: `{dev['device_id']}`\n")
    md_lines.append(f"- host: `{dev['host']}`\n")
    md_lines.append(f"- protocol: `{dev['protocol_version']}`\n")
    md_lines.append(f"- product: `{dev['product_key']}`\n")
    md_lines.append("\n| platform | dp_id | friendly_name | extras |\n")
    md_lines.append("|----------|-------|---------------|--------|\n")
    for e in dev["entities"]:
        extras_str = ", ".join(
            f"{k}={v}" for k, v in e.items()
            if k not in ("platform", "friendly_name", "id")
        )
        md_lines.append(f"| {e['platform']} | {e['id']} | {e['friendly_name']} | {extras_str} |\n")
    if dev["device_id"] in all_warnings:
        md_lines.append("\n**Warnings:**\n")
        for w in all_warnings[dev["device_id"]]:
            md_lines.append(f"- ⚠ {w}\n")

md_path = EXPORT_DIR / f"localtuya_{ts}.md"
md_path.write_text("".join(md_lines), encoding="utf-8")
print(f"MD   → {md_path}")

In [ ]:
# ── Print YAML to screen for inspection ───────────────────────────────────────
print(yaml.dump({"localtuya": localtuya_devices},
                allow_unicode=True, default_flow_style=False,
                sort_keys=False, indent=2))

## Troubleshooting

### dp_id shows `# FILL_DP_ID`
The Tuya spec endpoint (`/v1.0/iot-03/devices/{id}/specification`) did not return a
`dp_id` for that code. Options:
1. In the export notebook, check `data['devices']['spec']['<device_id>']` for what was returned.
2. Flash the device with the Tuya Smart app, observe the DPs in the debug view, or
   use `tinytuya wizard` which auto-discovers dp_ids over the local network.
3. Set the missing id manually in the patch cell above.

### Protocol version
Default is `3.3`. If a device does not respond:
- Try `3.1` for older devices.
- Try `3.4` for newer encrypted devices (e.g. some 2023+ Tuya devices).
Use the patch cell to override per device.

### Brightness / color-temp range
Tuya bulbs vary: some use 0-255, some use 10-1000. The notebook reads the `min`/`max`
from the product spec `values` field. If wrong, override `brightness_lower` /
`brightness_upper` in the patch cell.

### Power-monitoring scaling
The notebook computes `scaling` from the Tuya spec `scale` field plus a fixed
divisor (÷10 for power/voltage, ÷1000 for energy). If values look 10× off,
adjust the `scaling` field in the patch cell.

### Device not appearing in output
Check the **SKIPPED** section in the summary. Common reasons:
- `sub=True` — Zigbee/BLE sub-device; localtuya cannot control it directly (it goes through its gateway).
- no IP — device is offline at export time.
- no local_key — subscribe to the "Factory Info" API service in the Tuya IoT console.
- no recognised DPs — none of the device's DP codes are in `DP_TABLE`; extend the table.

### Adding DP codes not in the table
Look up the device's DPs in `data['devices']['spec']['<device_id>']`, then add an
entry to `DP_TABLE` in the classification cell and re-run.
